# FAD（CLAP music）— Colab GPU

ローカル CPU だと 1〜3 時間かかるので、ここ（GPU）で回す。

**事前準備（ローカル）**
1. `prttype/eval/fad/ref_wav` と `gen_wav` を zip する
2. Google Drive に置く（例: `MyDrive/fad_eval/fad_wav.zip`）

ランタイムは **GPU** を選ぶ。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# === 設定 ===
REPO_URL = "https://github.com/Kakeru0307/colab_train.git"
GITHUB_TOKEN = ""  # private なら入れる

# Drive 上の zip（中に ref_wav/ と gen_wav/ が入っている想定）
WAV_ZIP = "/content/drive/MyDrive/fad_eval/fad_wav.zip"
# または既に展開済みなら下を埋める（その場合 zip は使わない）
REF_DIR = ""
GEN_DIR = ""

JSON_OUT = "/content/drive/MyDrive/fad_eval/fad_result.json"

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/colab_train")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

url = REPO_URL
if GITHUB_TOKEN:
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
subprocess.run(["git", "clone", "--depth", "1", url, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "fadtk", "soundfile"], check=True)

import torch
print("cwd", Path.cwd())
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
from pathlib import Path
import zipfile
import shutil

WORK = Path("/content/fad_wav")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

ref = Path(REF_DIR) if REF_DIR else None
gen = Path(GEN_DIR) if GEN_DIR else None

if ref and gen and ref.is_dir() and gen.is_dir():
    print("using Drive dirs", ref, gen)
else:
    z = Path(WAV_ZIP)
    assert z.is_file(), f"zip がありません: {z}"
    with zipfile.ZipFile(z) as zf:
        zf.extractall(WORK)
    cands_ref = [p for p in WORK.rglob("ref_wav") if p.is_dir()]
    cands_gen = [p for p in WORK.rglob("gen_wav") if p.is_dir()]
    assert cands_ref and cands_gen, f"zip 内に ref_wav/gen_wav がありません: {list(WORK.iterdir())}"
    ref, gen = cands_ref[0], cands_gen[0]

n_ref = len(list(ref.glob("*.wav")))
n_gen = len(list(gen.glob("*.wav")))
print(f"ref={ref} ({n_ref})  gen={gen} ({n_gen})")
assert n_ref >= 1 and n_gen >= 1

In [ ]:
from pathlib import Path
import subprocess
import sys

out = Path(JSON_OUT)
out.parent.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable,
    "scripts/compute_fad.py",
    str(ref),
    str(gen),
    "--json-out",
    str(out),
]
print("+", " ".join(cmd))
subprocess.run(cmd, check=True)
print(out.read_text(encoding="utf-8"))